# L03 · 谱图（spectrogram）直觉——在学 FFT 之前先看结果

**学习目标**
1. 读懂谱图的三个轴（时间 × 频率 × 能量）
2. 在看图中区分纯音（pure tone）、和弦（chord）、扫频（frequency sweep）、噪声
3. 建立「谱图 = Module 5 学完后能亲手算出来的图」的直觉

> ⚠️ 本课**不推导任何公式**，只建立视觉直觉。
> FFT 和 STFT 的数学推导在 L37-L45。

← **上一课**　[L02 · 声音的数字表示](L02_sound_digital.ipynb)

> 上节课学习了 **声音的数字表示**：采样定理、PCM 数组、第一个可听正弦波。  
> 本课将探讨 **谱图直觉**。

## 1. 什么是谱图？

你听过乐队演奏吗？现场时，你能同时分辨出鼓、吉他和人声——哪怕它们同时响着。你的耳朵在实时做一件事：**把声音按频率分解，告诉你每个频率"有多响"**。

谱图（Spectrogram）做的正是这件事，只是它把结果画出来了：

```
纵轴 ↑  频率 (Hz)    高频 ─── 小提琴的高音、齿音
│
│       ████ ← 亮 = 这个频率能量高
│
│       ░░░░ ← 暗 = 这个频率几乎没有
│
低频 ─── 低音鼓、人声基频
└────────────────────→ 时间 (s)
```

三个轴的含义：
- **横轴**：时间——从左到右是声音的流逝
- **纵轴**：频率——从低（鼓声）到高（口哨）
- **颜色/亮度**：能量——越亮代表这个频率在这个时刻越响

读懂谱图，你就有了一双能"看见"声音的眼睛。后面的 FFT（L37）和 STFT（L43）就是计算谱图的数学机器——但先把图读懂，数学才有意义。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 工具：用 matplotlib.pyplot.specgram 显示谱图
# （L43–L45 将从零实现这背后的 STFT 算法；现在只需看图）
#
# 两个参数现在不用理解，L43 会细讲：
#   NFFT=256    每帧取 256 个采样点做 FFT → 频率分辨率 = sr/NFFT = 62.5 Hz/bin
#   noverlap=128  相邻帧重叠 128 点 → 每步跳 128 点（hop）= 8 ms，显示更连续（分辨率由 hop 决定，不是 overlap 本身）
def show_spec(signal, sr, title, ax, ylim=4000):
    ax.specgram(signal, Fs=sr, cmap='inferno', NFFT=256, noverlap=128)
    ax.set_ylim(0, ylim)
    ax.set_xlabel('时间 (s)'); ax.set_ylabel('频率 (Hz)')
    ax.set_title(title)

# L02 里你实现的 make_sine(duration, sr, freq) 封装的正是下面这行 np.sin
# 本课直接用 np.sin 构造信号，让你把注意力放在「看图」上
sr = 16000
t  = np.linspace(0, 2, 2 * sr, endpoint=False)

# 纯音：440 Hz 正弦波
pure_tone = np.sin(2 * np.pi * 440 * t)

fig, ax = plt.subplots(figsize=(8, 3))
show_spec(pure_tone, sr, '纯音 440 Hz（A4）', ax)
plt.tight_layout(); plt.show()
print('观察：一条水平亮线，位于 440 Hz，贯穿整个时间轴。')

### 读图：纯音的特征

- **一条水平亮线** = 整段时间内只有一个频率
- 线的位置 = 频率（440 Hz ≈ Y 轴 440 处）
- 线的亮度 = 能量（全程不变 = 振幅恒定）

你 6 个月后能从这张图的数学原理开始，推导出 `plt.specgram` 内部的每一步计算。

In [ ]:
# 和弦：三个频率同时响
chord = (
    np.sin(2 * np.pi * 261.6 * t) +   # C4
    np.sin(2 * np.pi * 329.6 * t) +   # E4
    np.sin(2 * np.pi * 392.0 * t)     # G4
) / 3.0

fig, ax = plt.subplots(figsize=(8, 3))
show_spec(chord, sr, '和弦 C-E-G（Do-Mi-Sol）', ax)
plt.tight_layout(); plt.show()
print('观察：三条水平亮线，对应 C4=261 Hz, E4=330 Hz, G4=392 Hz。')

In [ ]:
# 扫频（glissando）：频率随时间线性升高
f0, f1 = 200, 3000  # 从 200 Hz 扫到 3000 Hz
sweep = np.sin(2 * np.pi * (f0 * t + (f1 - f0) * t**2 / (2 * t[-1])))  # 积分形式，瞬时频率 f(t)=f0+(f1-f0)t/T

fig, ax = plt.subplots(figsize=(8, 3))
show_spec(sweep, sr, '扫频 200→3000 Hz（glissando）', ax)
plt.tight_layout(); plt.show()
print('观察：一条斜线从低频向高频上升，就像演奏中的滑音。')

In [ ]:
# 白噪声：所有频率同时随机响
rng = np.random.default_rng(42)
noise = rng.standard_normal(len(t))

fig, ax = plt.subplots(figsize=(8, 3))
show_spec(noise, sr, '白噪声（所有频率随机分布）', ax)
plt.tight_layout(); plt.show()
print('观察：整张图均匀地亮——所有频率、所有时刻都有能量。')

In [ ]:
# 四种声音并排对比
signals = [
    (pure_tone, '纯音 440 Hz'),
    (chord,     '和弦 C-E-G'),
    (sweep,     '扫频 200→3000 Hz'),
    (noise,     '白噪声'),
]

fig, axes = plt.subplots(2, 2, figsize=(12, 7))
for ax, (sig, title) in zip(axes.flat, signals):
    show_spec(sig, sr, title, ax)
plt.suptitle('四种声音的谱图对比', fontsize=13)
plt.tight_layout(); plt.show()

## 2. 四种声音的"指纹"

声音学家说：每种声音都有自己的谱图"指纹"。看三分钟谱图，你就能认出它：

| 声音类型 | 谱图特征 | 类比 |
|---|---|---|
| 纯音（单频率）| 一条水平亮线 | 激光——能量集中在一个方向 |
| 和弦（多频率）| 多条平行水平线 | 彩虹——多种颜色叠加 |
| 扫频 | 一条斜线（从左下→右上）| 警报声从低到高爬升 |
| 白噪声 | 整幅图均匀亮 | 沙滩——所有频率同时随机响 |

**关键直觉**：谱图里的"亮度"就是能量密度。一根亮线意味着几乎所有能量都集中在那个频率，就像用放大镜把阳光聚到一点。

## 3. 语音谱图的特征

真实语音比上面复杂：

- **共振峰（Formants）**：声道的谐振频率，形成水平亮带，决定「哪个元音」
- **基频（F0 / Pitch）**：声带振动频率，男声约 100-200 Hz，女声约 180-300 Hz
- **清音/浊音**：浊音（a, e, i）有清晰的谐波结构；清音（s, f）更像噪声

Whisper 输入的就是这种谱图（Mel spectrogram 版本，L46-L47 实现）。

In [ ]:
# 模拟「元音序列」：用不同共振峰组合近似 a / e / i
def vowel_approx(t, f0, f1, f2):
    """用基频 + 两个共振峰合成近似元音。"""
    return (
        0.5 * np.sin(2 * np.pi * f0 * t) +
        0.3 * np.sin(2 * np.pi * f1 * t) +
        0.2 * np.sin(2 * np.pi * f2 * t)
    )

seg_dur = 0.6
sr = 16000
t_seg = np.linspace(0, seg_dur, int(seg_dur * sr), endpoint=False)

# 简化的共振峰参数（非精确语音合成，仅用于谱图演示）
vowel_a = vowel_approx(t_seg, 130,  700, 1200)
vowel_e = vowel_approx(t_seg, 130,  400, 2200)
vowel_i = vowel_approx(t_seg, 130,  300, 3000)
combined = np.concatenate([vowel_a, vowel_e, vowel_i])

fig, ax = plt.subplots(figsize=(9, 3))
show_spec(combined, sr, '模拟元音序列  a → e → i  （共振峰变化）', ax, ylim=4000)
plt.tight_layout(); plt.show()
print('观察：三段声音的高频共振带位置不同，这就是元音区分的关键。')

## ✏️ 练习：预测谱图形状

不运行代码，先用文字描述下面信号的谱图长什么样：

```python
sig_a = np.sin(2*np.pi*1000*t)  # 1000 Hz 纯音
sig_b = sig_a * np.linspace(1, 0, len(t))  # 振幅从 1 线性衰减到 0
sig_c = np.sin(2*np.pi*500*t) + np.sin(2*np.pi*1500*t)  # 两频率和
```

在下面 Markdown cell 里写预测，然后运行代码验证。

In [ ]:
# ✏️ 填写你的预测（运行这格之前写下答案，然后对照下一格的谱图）
predictions = {
    "sig_a_1000hz_pure": None,      # 描述 sig_a (1000 Hz 纯音) 的谱图形状
    "sig_b_decaying":    None,      # 描述 sig_b (1000 Hz 衰减) 的谱图形状
    "sig_c_sweep":       None,      # 描述 sig_c (200→3000 Hz 扫频) 的谱图形状
}

unfilled = [k for k, v in predictions.items() if v is None]
assert not unfilled, f'请先填写预测再继续：{unfilled}'
print('预测已记录，现在运行下一格看真实谱图，对比你的预测。')
for k, v in predictions.items():
    print(f'  {k}: {v}')

In [ ]:
sr = 16000
t  = np.linspace(0, 2, 2 * sr, endpoint=False)

sig_a = np.sin(2 * np.pi * 1000 * t)
sig_b = sig_a * np.linspace(1, 0, len(t))
sig_c = np.sin(2 * np.pi * 500 * t) + np.sin(2 * np.pi * 1500 * t)

fig, axes = plt.subplots(1, 3, figsize=(13, 3))
for ax, sig, title in zip(axes,
        [sig_a, sig_b, sig_c],
        ['sig_a：1000 Hz', 'sig_b：1000 Hz 衰减', 'sig_c：500 + 1500 Hz']):
    show_spec(sig, sr, title, ax)
plt.tight_layout(); plt.show()

In [ ]:
# 验证：plt.specgram 返回的频率轴确认 440 Hz 纯音峰位于正确 bin
sr = 16000
t  = np.linspace(0, 1, sr, endpoint=False)
pure_tone = np.sin(2 * np.pi * 440 * t)

# specgram 返回 (spectrum, freqs, t_bins, im)
fig, ax = plt.subplots(figsize=(6, 2))
spectrum, freqs, _, _ = ax.specgram(pure_tone, Fs=sr, NFFT=1024)
plt.close()

# 找峰值所在频率 bin
peak_bin = spectrum.mean(axis=1).argmax()
peak_freq = freqs[peak_bin]
print(f'峰值 bin 频率：{peak_freq:.1f} Hz（期望 ≈ 440 Hz）')
assert abs(peak_freq - 440) < 20, f'峰位偏差过大: {peak_freq:.1f} Hz'
print('✅ 440 Hz 纯音的谱图峰值落在正确频率 bin')


## 本课收束

你现在能**看懂**谱图，但还不知道它是**怎么算出来**的。

| 直觉 | 数学 | 课程位置 |
|---|---|---|
| 频率 = Y 轴位置 | DFT 的输出频点 | L37-L39 |
| 能量 = 颜色亮度 | 复数模的平方 | L40-L41 |
| 时间切片 = 窗函数（window function） | STFT 窗函数 | L36, L43-L45 |
| Mel 版谱图 | Mel filterbank | L46-L47 |

Module 5（L32-L53）结束时，你将能从零开始计算本课所有谱图——
包括 `plt.specgram` 内部的每一步，都会是你自己写的代码。

**下一课 L04**：正弦波三要素——频率、振幅、相位各控制什么，
掌握后再去看 L06 的欧拉公式，FFT 就有了具体的「砖头」。

## ✏️ 读谱图挑战：不看代码，只看图推断声音

**这是本课的核心练习。** 谱图比公式出现早——你在 L37 才会知道 FFT 怎么算，但你现在就要能读图。

---

**题目**：给你三幅谱图的文字描述，推断对应的声音类型：

**图A**：整幅图从上到下亮度均匀，没有特别亮的横线或斜线。
→ 这是什么声音？（纯音 / 和弦 / 扫频 / 白噪声）

**图B**：图中有三条水平亮线，分别在约 261 Hz、330 Hz、392 Hz 处，从左到右持续不变。
→ 这是什么声音？描述它的音乐含义。

**图C**：图中有一条从左下角（200 Hz）爬升到右上角（3000 Hz）的斜亮线。
→ 这是什么声音？在音乐中对应什么演奏技法？

**图D**：图中有水平亮线，但亮度从左到右逐渐变暗，最后几乎消失。
→ 这是什么声音？

---

把你的答案写在下面验证格的字典里，代码会告诉你对不对。

In [ ]:
# ✏️ 把答案填入字典，然后运行对答案
import numpy as np

quiz_answers = {
    "图A": None,  # 填 '白噪声' / '纯音' / '和弦' / '扫频'
    "图B": None,  # 填 '和弦' + 说明（如 'C大调三和弦'）
    "图C": None,  # 填 '扫频' + 说明
    "图D": None,  # 填 '衰减纯音' 或描述
}

unfilled = [k for k, v in quiz_answers.items() if v is None]
assert not unfilled, f'还未填写：{unfilled}'

# 验证：答案应包含关键词
correct = {
    "图A": ["白噪声", "noise", "随机"],
    "图B": ["和弦", "chord", "多频", "三和弦"],
    "图C": ["扫频", "sweep", "glissando", "上行"],
    "图D": ["衰减", "decay", "envelope", "减弱"],
}

all_correct = True
for fig, keywords in correct.items():
    ans = str(quiz_answers.get(fig, "")).lower()
    kw_lower = [k.lower() for k in keywords]
    if any(k in ans for k in kw_lower):
        print(f'  {fig} ✅')
    else:
        print(f'  {fig} ⚠️  参考答案包含：{keywords}')
        all_correct = False

if all_correct:
    print('\n🎉 谱图读图挑战通过！你已经能用眼睛"听"声音了。')
else:
    print('\n💡 复习 §2 规律总结，再对照谱图图例，修改答案后重新运行。')

In [ ]:
# ✏️ 本课自评
l03_review = {
    "three_axes_understood":   None,  # 能说出谱图三轴含义（时间/频率/能量）？True/False
    "four_patterns_recognized": None, # 能识别四种声音的谱图指纹？True/False
    "prediction_exercise_done": None, # 预测练习在运行前填写了？True/False
    "quiz_passed":              None, # 读谱图挑战通过？True/False
}

unfilled = [k for k, v in l03_review.items() if v is None]
assert not unfilled, f'还未填写：{unfilled}'
weak = [k for k, v in l03_review.items() if v is False]
if weak:
    print(f'⚠️  需要加强：{weak}')
else:
    print('✅ L03 全部通关！进入 L04：正弦波三要素')

---

→ **下一课**　[L04 · 正弦波三要素](../1_complex_trig/L04_trig.ipynb)

> 下节课将学习 **正弦波三要素**：频率决定音高、振幅决定响度、相位决定起点，亲手实现。